In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.ipa_processing import syllabify_ipa
SEP = "."

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])

In [3]:
df

,syllable1_start,syllable1_end,syllable2_start,syllable2_end,syllable1,syllable2,context,word,word_id,art_f,ac_f,subject
0,1.203764,1.353764,1.353764,1.653764,m_æ,s_k_ɝ,m_æ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
1,1.353764,1.653764,1.653764,1.803764,s_k_ɝ,eɪ_d,m_æ.s_k_ɝ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
2,1.833764,2.023764,2.023764,2.253764,p_ɑ_ɹ,t_i_z,p_ɑ_ɹ,parties,1.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
3,2.963764,3.023764,3.023764,3.203764,ɪ,m_æ,ɪ,imagination,4.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
4,3.023764,3.203764,3.203764,3.353764,m_æ,dʒ_ɪ,ɪ.m_æ,imagination,4.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
8064,23.009070,23.059070,23.059070,23.289070,ʌ,k_w_ɑ,ʌ,aquatic,34.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
8065,23.059070,23.289070,23.289070,23.449070,k_w_ɑ,t_ɪ_k,ʌ.k_w_ɑ,aquatic,34.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
8066,23.449070,23.639070,23.639070,23.729070,k_ɑ_m,p_ʌ,k_ɑ_m,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
8067,23.639070,23.729070,23.729070,23.919070,p_ʌ,t_ɪ,k_ɑ_m.p_ʌ,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1


In [4]:
with open("../../../phone_based/pronunciation_counts/pronunciation_counts_ae.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="ae"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for syllable1, syllable2 in tqdm.tqdm(zip(df["syllable1"], df["syllable2"]), total=len(df)):
    bigrams.add(syllable1+SEP+syllable2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    syllables = pronunciation.split(SEP)
    for syllable in syllables:
        if syllable not in monogram_counts:
            monogram_counts[syllable] = 0
        monogram_counts[syllable] += count

100%|██████████| 48576/48576 [00:00<00:00, 1559572.19it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 48576/48576 [00:00<00:00, 300832.19it/s]


In [7]:
df["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["syllable1"] + SEP + x["syllable2"]), axis=1)
df["syllable2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1)

In [8]:
df

,syllable1_start,syllable1_end,syllable2_start,syllable2_end,syllable1,syllable2,context,word,word_id,art_f,ac_f,subject,bigram_surprisal,syllable2_pos_entropy
0,1.203764,1.353764,1.353764,1.653764,m_æ,s_k_ɝ,m_æ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1,13.296792,3.112846
1,1.353764,1.653764,1.653764,1.803764,s_k_ɝ,eɪ_d,m_æ.s_k_ɝ,masquerade,0.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1,7.661527,0.673012
2,1.833764,2.023764,2.023764,2.253764,p_ɑ_ɹ,t_i_z,p_ɑ_ɹ,parties,1.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1,9.370866,2.086496
3,2.963764,3.023764,3.023764,3.203764,ɪ,m_æ,ɪ,imagination,4.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1,8.504313,3.766331
4,3.023764,3.203764,3.203764,3.353764,m_æ,dʒ_ɪ,ɪ.m_æ,imagination,4.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1,5.266319,1.168747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8064,23.009070,23.059070,23.059070,23.289070,ʌ,k_w_ɑ,ʌ,aquatic,34.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1,13.201482,3.274454
8065,23.059070,23.289070,23.289070,23.449070,k_w_ɑ,t_ɪ_k,ʌ.k_w_ɑ,aquatic,34.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1,9.701221,0.000000
8066,23.449070,23.639070,23.639070,23.729070,k_ɑ_m,p_ʌ,k_ɑ_m,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1,9.384769,2.143118
8067,23.639070,23.729070,23.729070,23.919070,p_ʌ,t_ɪ,k_ɑ_m.p_ʌ,competition,35.0,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1,1.671659,1.152917
